# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.
> **Run this notebook in Colab, not locally.** It queries the gated Hugging Face warehouse
> release with a personal READ token (stored as a Colab Secret named `HF_TOKEN`, never pasted
> into a cell). Open in Colab, run top to bottom, then download/commit the executed `.ipynb`.

## 1. The contract, in plain words

**Unit of analysis:** one row = one content item (page), aggregated over a fixed calendar
month from the daily fact table. Not a page-day — I'm rolling the month up to one row per
`content_hash_id` before I ever look at a label.

**Table(s):** `fact_content_daily_performance` (partition `month=2026-03`) for the daily
search/engagement numbers, joined to `dim_content` for static content metadata
(`word_count`, `content_type`, age). `dim_clients` is used only to check panel coverage
(`gsc_data_start`, `ga4_data_start`) — never joined in as a feature source.

**Time window:** March 2026 (`month=2026-03`) — a mid-panel month, not the final month. The
`_sample` table *is* the final month (June 2026) per the warning on this card, so I'm treating
June as a sealed test month and iterating here instead.

**What I'd predict/rank:** a proxy label built fresh from this month's own daily data —
"declining" defined as impressions in the back half of March being meaningfully lower than the
front half (mirrors the starter CSV's `trend_direction` idea, but computed from real daily rows
instead of a precomputed column). The output is a rank/score per content item, same shape as
Lane 2's actual deliverable.

**One thing I deliberately exclude:** `fact_content_query_90d`. Its window is a fixed 90-day
span covering the snapshot's most recent ~3 months, not March — joining it here would either
silently misalign windows or pull in information from months after the one I'm scoring. I'm
leaving it out until I explicitly solve that alignment, rather than guess.

In [1]:
%pip -q install duckdb huggingface_hub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, getpass

# Colab Secrets panel (key icon) is the safe way to set this - never paste the token in a cell.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':  f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':  f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':   f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}
for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:12} {n:>12,} rows')

dim_clients           104 rows
dim_content       519,606 rows
fact_daily      9,841,378 rows


## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `word_count` (dim_content) | Feature | Static content property, known long before March even starts |
| `content_age_days` / age tier (dim_content) | Feature | Known at any decision moment — content already exists |
| `gsc_avg_position`, first 15 days of March | Feature | Computed only from days *before* the outcome window I'm scoring |
| `gsc_impressions`, `gsc_clicks`, first 15 days | Feature | Same — front-half-of-month only, never the back half |
| GA4 engagement fields | Feature, **filtered** | Only used where `ga4_data_available IS TRUE` — zero-fill rows before a client's `ga4_data_start` are not real zeros |
| Back-half-of-March impressions | Label / proxy | This is exactly what the "declining" proxy is computed from — never a feature |
| `content_hash_id`, `client_hash_id`, `report_date` | Context | Grouping, joining, and the grain check only — never model inputs |
| `fact_content_query_90d` (whole table) | Excluded | Window doesn't align with March 2026 — see contract section 1 |
| GA4 columns where `ga4_data_available` is FALSE | Excluded | Zero-filled placeholder, not observed zero engagement |

In [3]:
# nothing to run here - section 2 is a classification table, verified by the queries in section 3

## 3. Verify it with queries

Three claims from the contract above, each checked with a real query against `month=2026-03`.

In [4]:
# Query 1 - GRAIN: one row of the daily fact really is one content-item-day.
# Zero rows back means the grain holds.
grain_check = con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) AS c
    FROM {TABLES['fact_daily']}
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Duplicate (content, day) rows found: {len(grain_check)}  (0 = grain holds)")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (content, day) rows found: 0  (0 = grain holds)


,content_hash_id,report_date,c


In [5]:
# Query 2 - COUNTS + DATE SPAN of my March slice.
counts = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT content_hash_id) AS distinct_content_items,
           COUNT(DISTINCT client_hash_id) AS distinct_clients,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
""").df()
counts

,total_rows,distinct_content_items,distinct_clients,min_date,max_date
0,9841378,331437,55,2026-03-01,2026-03-31


In [6]:
# Query 3 - AVAILABILITY, filtered with IS TRUE (per the flyrank-data skill's panel warning:
# rows before a client's ga4_data_start are zero-filled, not real zero engagement).
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_available
    FROM {TABLES['fact_daily']}
""").df()
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_available
0,9841378,413966,4.2


### Five features (max), each with an "available when?" line

Built only from the **first 15 days of March** — the back half is reserved for the label, so
no feature here can see into the outcome window it will later be scored against.

1. **`word_count`** — available when? Always. It's a static content property published before
   March even begins.
2. **`days_since_last_update_at_mar1`** — available when? At the start of March, since it only
   depends on content history up to that date, not on anything inside the window being scored.
3. **`avg_position_first15`** — available when? By March 15, computed only from `report_date`
   in the first half of the month.
4. **`impressions_first15`** — available when? By March 15, same front-half-only window.
5. **`ctr_first15`** (`clicks_first15 / impressions_first15`) — available when? By March 15 —
   an early-window efficiency signal, never touching the back-half outcome data.

In [7]:
features = con.sql(f"""
    SELECT
        f.content_hash_id,
        ANY_VALUE(c.word_count)                AS word_count,
        ANY_VALUE(DATE '2026-03-01' - c.last_optimized_date) AS days_since_last_update_at_mar1,
        AVG(CASE WHEN f.report_date < DATE '2026-03-16' THEN f.gsc_avg_position END)   AS avg_position_first15,
        SUM(CASE WHEN f.report_date < DATE '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS impressions_first15,
        SUM(CASE WHEN f.report_date < DATE '2026-03-16' THEN f.gsc_clicks ELSE 0 END)      AS clicks_first15,
        -- kept separate, back half only, used for the label below - NOT a feature
        SUM(CASE WHEN f.report_date >= DATE '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS impressions_second15
    FROM {TABLES['fact_daily']} f
    JOIN {TABLES['dim_content']} c USING (content_hash_id)
    GROUP BY f.content_hash_id
    HAVING impressions_first15 >= 50   -- need a measurable front half to build a ratio at all
""").df()

features['ctr_first15'] = features['clicks_first15'] / features['impressions_first15']
print(f"{len(features):,} content items with a usable first-15-day window")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

92,548 content items with a usable first-15-day window


,content_hash_id,word_count,days_since_last_update_at_mar1,avg_position_first15,impressions_first15,clicks_first15,impressions_second15,ctr_first15
0,content_e70ab7e3c46c295c,<NA>,<NA>,3.435425,252.0,1.0,353.0,0.003968
1,content_b867ac9f5c0b11d6,3030,<NA>,51.002086,460.0,1.0,454.0,0.002174
2,content_c94498274045e924,<NA>,<NA>,48.553444,1461.0,1.0,1468.0,0.000684
3,content_616d0730a879c14e,2595,-86,43.894622,1600.0,1.0,2205.0,0.000625
4,content_b7e2a289753f5934,3350,-80,30.374051,2186.0,1.0,2809.0,0.000457


### 4. The trap — one deliberate leak, on purpose

Define the proxy label from the back half of March (the window I excluded from every feature
above), then watch what happens if I "accidentally" hand the model a column computed from
that same back half.

In [8]:
features['is_declining_proxy'] = (
    features['impressions_second15'] < 0.8 * features['impressions_first15']
).astype(int)
print(f"Proxy label base rate: {features['is_declining_proxy'].mean():.3f}")

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_cols = ['word_count', 'days_since_last_update_at_mar1', 'avg_position_first15',
               'impressions_first15', 'ctr_first15']
model_data = features.dropna(subset=honest_cols + ['is_declining_proxy'])
X, y = model_data[honest_cols], model_data['is_declining_proxy']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f"HONEST auc (front-half features only): {honest_auc:.3f}")

Proxy label base rate: 0.286
HONEST auc (front-half features only): 0.571


In [9]:
# THE TRAP: add a column derived straight from the back half - the exact window the label
# is built from - and watch the score jump toward perfect.
leaky_data = model_data.copy()
leaky_data['impressions_second15_LEAK'] = features.loc[leaky_data.index, 'impressions_second15']

leaky_cols = honest_cols + ['impressions_second15_LEAK']
X_leak, y_leak = leaky_data[leaky_cols], leaky_data['is_declining_proxy']
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_leak, y_leak, test_size=0.25, random_state=42, stratify=y_leak)

leaky_model = LogisticRegression(max_iter=1000).fit(X_tr2, y_tr2)
leaky_auc = roc_auc_score(y_te2, leaky_model.predict_proba(X_te2)[:, 1])
print(f"LEAKY auc (back-half column included): {leaky_auc:.3f}")
print(f"\nJump from leakage: {honest_auc:.3f} -> {leaky_auc:.3f}")
print("Deleting the leaked column now and keeping only the honest number above.")

# delete it - keep the honest model/number as the real result
del leaky_data, leaky_cols, X_leak, y_leak, leaky_model

LEAKY auc (back-half column included): 1.000

Jump from leakage: 0.571 -> 1.000
Deleting the leaked column now and keeping only the honest number above.


## 4. Data limits — one named limitation

**This slice can't tell me whether a decline is seasonal.** March 2026 is one month with no
year-over-year comparison available in this snapshot's history depth for most clients (per
`dim_clients.gsc_data_start`, many clients don't have 12+ months back from March). A page that
dips every March for a predictable seasonal reason looks identical, in this contract, to a page
genuinely losing relevance — the daily fact table has no seasonal baseline to separate the two.
Any "declining" flag built here is a within-month momentum signal, not a claim about the
underlying cause.

In [10]:
# check panel depth relative to March, to size the seasonal-blindness limitation honestly
panel_depth = con.sql(f"""
    SELECT
        COUNT(*) AS n_clients,
        COUNT(*) FILTER (WHERE gsc_data_start <= DATE '2025-03-01') AS clients_with_yoy_history
    FROM {TABLES['dim_clients']}
""").df()
panel_depth

,n_clients,clients_with_yoy_history
0,104,3


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.